In this whole process, i will think in terms of industrial scale and i am trying to build the whole LLM pipeline from scratch to understand everything better. In this notebook we start with the pre processing step. The first thing we have is data. I am actually following @ahmadosman 's step-by-step LLM Engineering projects so first to understand the data thing in better i will do the 21st project and 22 nd projects which are about 

21st project - Build a pre-training data pipeline :
Data quality is one of the highest impacts part of the stack. Finweb Released 15-trillion-tokens dataset built from Common Crawl snapshots with filtering and de-duplication choices aimed at strong open pretraining.Dolma provided the open corpus behind OLMo

-> Build :
Document Ingestion
Language ID
Deduplication
Quality filtering
Toxicity or policy filtering
PII Detection
Train/validation/Test splitting
Contamination checks

-> Measure :
Duplication rate
Token distribution by source
Language mix
Average Document Length
Per-domain Validation Loss
Benchmark Contamination 

-> Break :
Train with Duplicates
Train on low-quality scraped text
Leak evaluation data into training
Over-filter until the corpus loses diversity
Under-filter until the model memorizes junk

Core Insight - Model quality is often data-quality wearing a model-size costune

22nd project - Synthetic data: Generate, filter, and prove it helped :
Synthetic data can help when it is targeted, filtered, and evaluated honestly. The phi-1 work showed that small models trained on high-quality textbook-like data and synthetic code data could perform suprisingly well on coding benchmarks.

-> Build :
A synthetic instruction generator
A verifier or quality classifier
A dedepulicator
A difficulty curriculum
Eval splits kept isolated from generation prompts

-> Measure :
Real-only vs Synthetic-only versus mixed training
Learning curves
Diversity
Rejection rate
Error propagation
Downstream task performance

-> Break :
Train on unfiltered synthetic data
Generate examples from the same benchmark you evaluate on 
Use one teacher model for everything
Add synthetic data after performance saturates.

Core Insight - Synthetic data is not free intelligence. It is data-engineering and evaluation problem.

Hey let's understand all of them in detail one by one and build everything from scratch don't worry!


Okay there is a clear description of how fineweb was created which we will kind of replicate in the process of learning all the things mentioned in the projects

What is document ingestion first of all? Let's search on the internet what this is exactly. I feel like there are bunch of definations regarding this on the internet what's the thing that i need to build when ahmad osman says to build a document ingestion?. Let's ask GPT what that means exaclty 
In context of pre-training LLM document ingestion means that Getting raw documents from the source dataset where the pre-processing pipeline can operate on them.
Before that first let's understand in brief how Commoncrawl works and what it stores and what it is exactly?

Common Crawl:
-> It is a non-profit organization that has been crawling web since 2008
-> It is a massive open source library of the internet, Every month it's CCBot scans billions of pages and archieves them
-> Commoncrawl offers data in three formats : WARC(web ARChive) Files (The rawest form containing full HTTP responses, Perfect if you need images, HTML Parsing, or complete Page reconstruction), WAT (Web ARChive Transformation) files (They are like summaries of WARC files, They contain metadata in JSON format, such as all links on the page, HTTP headers etc), WET (Web extracted text files) (Plain text extracted from the WARC files (no HTML, no media)) ideal for NLP or training based models 

let's inspect how it looks inside common crawl's archieves. 


In [43]:
import requests
import json
from warcio.archiveiterator import ArchiveIterator

Before fetching any page, we need to know which crawl (index) it belongs to. Common crawl organizes it's data into periodic crawls, for examples CC-MAIN-2025-33 means it's 33rd crawl of 2025.
So first we will fetch list of available indexes

In [44]:
def get_available_indexes():
    collections_url="https://index.commoncrawl.org/collinfo.json"
    response=requests.get(collections_url,timeout=30)
    collections=response.json()
    cdx_indices=[]
    for collection in collections:
        if "cpx-api" in collection:
            cdx_indices.append(collection["cpx-api"])
    cdx_indices.sort(reverse=True)
    return cdx_indices

I will work on the latest crawl CC-MAIN-2026-25 Crawl
But my doubt is which one should i consider WARC? WAT? or WET? Let's find out why
Fineweb actually starts with WARC but why don't we do this? we can directly start the pipleine from WET right? Because we want to have a better control. So i have downloaded warc.paths.gz into my desktop now. and i have inspected head warc.paths it actually contains the paths to 1,00,000 files in this WARC. Let's download one warc file

In [45]:
with open("/Users/vedasamhith/Desktop/warc.paths","r") as f:
    first_path=f.readline().strip()
print(first_path)

crawl-data/CC-MAIN-2026-25/segments/1780687572080.85/warc/CC-MAIN-20260605214811-20260606004811-00000.warc.gz


In [46]:
base_url="https://data.commoncrawl.org/"
download_url = base_url + first_path
print(download_url)

https://data.commoncrawl.org/crawl-data/CC-MAIN-2026-25/segments/1780687572080.85/warc/CC-MAIN-20260605214811-20260606004811-00000.warc.gz


In [47]:
import requests
response=requests.get(download_url,stream=True)
with open("sample_warc.gz","wb") as f:
    for chunk in response.iter_content(chunk_size=65536):
        if chunk:
            f.write(chunk)


In [48]:
import gzip
with gzip.open("sample_warc.gz","rb") as f:
    data=f.read()

In [49]:
idx=data.find(b"WARC-Type: response")
print(idx)

1482


In [50]:
idx_start=data.rfind(b"WARC/1.0",0,idx)
print(idx_start)

1472


In [51]:
print(data[idx_start:idx_start+20])

b'WARC/1.0\r\nWARC-Type:'


In [52]:
idx_end=data.find(b"WARC/1.0",idx_start+1)
print(idx_end)

19571


In [53]:
record=data[idx_start:idx_end]

In [54]:
print(record[:700])

b'WARC/1.0\r\nWARC-Type: response\r\nWARC-Date: 2026-06-05T22:41:22Z\r\nWARC-Record-ID: <urn:uuid:bc1d9f66-f584-4630-9648-94691fecba8e>\r\nContent-Length: 17463\r\nContent-Type: application/http; msgtype=response\r\nWARC-Warcinfo-ID: <urn:uuid:b7b9eeb0-5a36-4c28-bb90-5234268f18cf>\r\nWARC-Concurrent-To: <urn:uuid:a56c616b-d5f5-410d-95dd-88835ad5bf62>\r\nWARC-IP-Address: 103.30.42.70\r\nWARC-Target-URI: http://0530jtss.com/content.asp?id=8198&bg=186\r\nWARC-Protocol: http/1.1\r\nWARC-Payload-Digest: sha1:4F2MWZRF5AHPM25IFVU2RNF7Q4DK2PKW\r\nWARC-Block-Digest: sha1:N5D3UH7JJWW725CERID53II374GGJKT2\r\nWARC-Identified-Payload-Type: application/xhtml+xml\r\n\r\nHTTP/1.1 200 OK\r\nCache-Control: private\r\nContent-Type: text/html\r\nX-'


In [55]:
print(record)

b'WARC/1.0\r\nWARC-Type: response\r\nWARC-Date: 2026-06-05T22:41:22Z\r\nWARC-Record-ID: <urn:uuid:bc1d9f66-f584-4630-9648-94691fecba8e>\r\nContent-Length: 17463\r\nContent-Type: application/http; msgtype=response\r\nWARC-Warcinfo-ID: <urn:uuid:b7b9eeb0-5a36-4c28-bb90-5234268f18cf>\r\nWARC-Concurrent-To: <urn:uuid:a56c616b-d5f5-410d-95dd-88835ad5bf62>\r\nWARC-IP-Address: 103.30.42.70\r\nWARC-Target-URI: http://0530jtss.com/content.asp?id=8198&bg=186\r\nWARC-Protocol: http/1.1\r\nWARC-Payload-Digest: sha1:4F2MWZRF5AHPM25IFVU2RNF7Q4DK2PKW\r\nWARC-Block-Digest: sha1:N5D3UH7JJWW725CERID53II374GGJKT2\r\nWARC-Identified-Payload-Type: application/xhtml+xml\r\n\r\nHTTP/1.1 200 OK\r\nCache-Control: private\r\nContent-Type: text/html\r\nX-Crawler-Content-Encoding: gzip\r\nVary: Accept-Encoding\r\nServer: Microsoft-IIS/7.5\r\nSet-Cookie: ASPSESSIONIDAABRRCBR=OEOHOPJBKNLJACAIJJHFLOJA; path=/\r\nX-Powered-By: WAF/2.0\r\nDate: Fri, 05 Jun 2026 22:41:22 GMT\r\nX-Crawler-Content-Length: 5985\r\nContent

In [56]:
idx_first_space=record.find(b"\r\n\r\n")
print(idx_first_space)

628


In [57]:
warc_header_bytes=record[:idx_first_space]

In [58]:
idx_second_space=record.find(b"\r\n\r\n",idx_first_space+4)
print(idx_second_space)

962


In [59]:
http_header_bytes=record[idx_first_space+4:idx_second_space]
html_bytes=record[idx_second_space+4:]

In [60]:
warc_dict={}
http_dict={}

In [61]:
warc_text=warc_header_bytes.decode('utf-8',errors="replace")
for line in warc_text.splitlines():
    if ':' in line:
        key,value=line.split(":",1)
        warc_dict[key.strip()]=value.strip()

http_text=http_header_bytes.decode('utf-8',errors="replace")
for line in http_text.splitlines():
    if ':' in line:
        key,value=line.split(':',1)
        http_dict[key.strip()]=value.strip()

In [62]:
print(http_dict)

{'Cache-Control': 'private', 'Content-Type': 'text/html', 'X-Crawler-Content-Encoding': 'gzip', 'Vary': 'Accept-Encoding', 'Server': 'Microsoft-IIS/7.5', 'Set-Cookie': 'ASPSESSIONIDAABRRCBR=OEOHOPJBKNLJACAIJJHFLOJA; path=/', 'X-Powered-By': 'WAF/2.0', 'Date': 'Fri, 05 Jun 2026 22:41:22 GMT', 'X-Crawler-Content-Length': '5985', 'Content-Length': '17129'}


In [63]:
idx_html_start=html_bytes.find(b"charset=")
i=idx_html_start+len(b"charset=")
terminator={ord('"'),ord("'"),ord(">"),ord(";")}
while html_bytes[i] not in terminator:
    i+=1
idx_html_end=i

encoding=html_bytes[idx_html_start+len(b"charset="):idx_html_end]
print(encoding)

b'gb2312'


In [64]:
print(warc_dict)

{'WARC-Type': 'response', 'WARC-Date': '2026-06-05T22:41:22Z', 'WARC-Record-ID': '<urn:uuid:bc1d9f66-f584-4630-9648-94691fecba8e>', 'Content-Length': '17463', 'Content-Type': 'application/http; msgtype=response', 'WARC-Warcinfo-ID': '<urn:uuid:b7b9eeb0-5a36-4c28-bb90-5234268f18cf>', 'WARC-Concurrent-To': '<urn:uuid:a56c616b-d5f5-410d-95dd-88835ad5bf62>', 'WARC-IP-Address': '103.30.42.70', 'WARC-Target-URI': 'http://0530jtss.com/content.asp?id=8198&bg=186', 'WARC-Protocol': 'http/1.1', 'WARC-Payload-Digest': 'sha1:4F2MWZRF5AHPM25IFVU2RNF7Q4DK2PKW', 'WARC-Block-Digest': 'sha1:N5D3UH7JJWW725CERID53II374GGJKT2', 'WARC-Identified-Payload-Type': 'application/xhtml+xml'}


In [65]:
print(http_dict)

{'Cache-Control': 'private', 'Content-Type': 'text/html', 'X-Crawler-Content-Encoding': 'gzip', 'Vary': 'Accept-Encoding', 'Server': 'Microsoft-IIS/7.5', 'Set-Cookie': 'ASPSESSIONIDAABRRCBR=OEOHOPJBKNLJACAIJJHFLOJA; path=/', 'X-Powered-By': 'WAF/2.0', 'Date': 'Fri, 05 Jun 2026 22:41:22 GMT', 'X-Crawler-Content-Length': '5985', 'Content-Length': '17129'}


In [24]:
print(http_header_bytes)

b'HTTP/1.1 200 OK\r\nCache-Control: private\r\nContent-Type: text/html\r\nX-Crawler-Content-Encoding: gzip\r\nVary: Accept-Encoding\r\nServer: Microsoft-IIS/7.5\r\nSet-Cookie: ASPSESSIONIDAABRRCBR=OEOHOPJBKNLJACAIJJHFLOJA; path=/\r\nX-Powered-By: WAF/2.0\r\nDate: Fri, 05 Jun 2026 22:41:22 GMT\r\nX-Crawler-Content-Length: 5985\r\nContent-Length: 17129'


In [66]:
idx_http_start=http_header_bytes.find(b"\r\n")
line=http_header_bytes[:idx_http_start].decode('utf-8').split(' ')
http_status=line[1]
print(http_status)


200


In [67]:

record_dict={}
record_dict["url"]=warc_dict["WARC-Target-URI"]
record_dict["timestamp"]=warc_dict["WARC-Date"]
record_dict["warc_record_id"]=warc_dict["WARC-Record-ID"]
record_dict["http_status"]=http_status
record_dict["content_type"]=http_dict["Content-Type"]
record_dict["charset"]=encoding.decode('utf-8',errors="replace")
record_dict["html"]=html_bytes.decode(encoding.decode('utf-8',errors="replace"),errors="replace")



In [68]:
print(record_dict)

{'url': 'http://0530jtss.com/content.asp?id=8198&bg=186', 'timestamp': '2026-06-05T22:41:22Z', 'warc_record_id': '<urn:uuid:bc1d9f66-f584-4630-9648-94691fecba8e>', 'http_status': '200', 'content_type': 'text/html', 'charset': 'gb2312', 'html': '<link href="css.css" rel="stylesheet" type="text/css" />\r\n<script Language="JavaScript" src="12.js"></script>\r\n<script language="javascript">\r\n<!--\r\nfunction ImgAutoSize(img) \r\n{ \r\nvar MaxWidth=550;//设置图片宽度界限 \r\nvar MaxHeight=550;//设置图片高度界限 \r\nvar HeightWidth=img.offsetHeight/img.offsetWidth;//设置高宽比 \r\nvar WidthHeight=img.offsetWidth/img.offsetHeight;//设置宽高比 \r\nif(img.readyState!="complete")return false;//确保图片完全加载 \r\nif(img.offsetWidth>MaxWidth){ \r\nimg.width=MaxWidth; \r\nimg.height=MaxWidth*HeightWidth; \r\n} \r\nif(img.offsetHeight>MaxHeight){ \r\nimg.height=MaxHeight; \r\nimg.width=MaxHeight*WidthHeight; \r\n} \r\n}\r\n//-->\r\n</script>\r\n<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR

In [69]:
len(data)

4030350589

Okay upto now i have did for single record now let me generalise the process for one whole WARC file 

In [70]:
idx=0
idx_start=0
idx_end=0
document_ingestion=[]
while idx < len(data):
    idx=data.find(b"WARC-Type: response",idx_end)
    if idx==-1:
        break
    idx_start=data.rfind(b"WARC/1.0",idx_end,idx)
    idx_end=data.find(b"WARC/1.0",idx_start+len(b"WARC/1.0"))
    record=data[idx_start:idx_end]
    idx_first_space=record.find(b"\r\n\r\n")
    warc_header_bytes=record[:idx_first_space]
    idx_second_space=record.find(b"\r\n\r\n",idx_first_space+4)
    http_header_bytes=record[idx_first_space+4:idx_second_space]
    html_bytes=record[idx_second_space+4:]
    warc_dict={}
    http_dict={}

    warc_text=warc_header_bytes.decode('utf-8',errors="replace")
    for line in warc_text.splitlines():
        if ':' in line:
            key,value=line.split(":",1)
            warc_dict[key.strip()]=value.strip()

    http_text=http_header_bytes.decode('utf-8',errors="replace")
    for line in http_text.splitlines():
        if ':' in line:
            key,value=line.split(':',1)
            http_dict[key.strip().lower()]=value.strip()

    idx_html_start=html_bytes.find(b"charset=")
    if idx_html_start!=-1:
        i=idx_html_start+len(b"charset=")
        terminator={ord('"'),ord("'"),ord(">"),ord(";")}
        while html_bytes[i] not in terminator:
            i+=1
        idx_html_end=i
        decoder=html_bytes[idx_html_start+len(b"charset="):idx_html_end].decode('utf-8',errors="replace").strip()
        if decoder=="":
            decoder="utf-8"
    else:
        decoder='utf-8'
    idx_http_start=http_header_bytes.find(b"\r\n")
    line=http_header_bytes[:idx_http_start].decode('utf-8').split(' ')
    http_status=line[1]
    record_dict={}
    record_dict["url"]=warc_dict["WARC-Target-URI"]
    record_dict["timestamp"]=warc_dict["WARC-Date"]
    record_dict["warc_record_id"]=warc_dict["WARC-Record-ID"]
    record_dict["http_status"]=int(http_status)
    record_dict["content_type"]=http_dict.get("content-type","")
    record_dict["charset"]=decoder
    try:
        html=html_bytes.decode(decoder,errors="replace")
    except LookupError:
        html=html_bytes.decode('utf-8',errors="replace")
    record_dict["html"]=html
    document_ingestion.append(record_dict)



with open("document_ingestion.jsonl","w",encoding="utf-8") as f:
    for record in document_ingestion:
        json.dump(record,f,ensure_ascii=False)
        f.write("\n")


Okay so i am done with the document ingestion thing let's look at document_ingestion.jsonl and see how a single entity from it looks like, the josnl file is around 4.21GB, i know that there will be a better way to do all of these but if i keep on optimizing on one thing, i can literally spend my whole time on that, but that's not the goal, the goal is to understand every part

In [71]:
import json

with open("document_ingestion.jsonl","r",encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i==6000:
            record=json.loads(line)
            print(record["html"])
            break

<!doctype html>
<!--[if lt IE 7]><html class="no-js lt-ie9 lt-ie8 lt-ie7" lang="en-US" prefix="og: https://ogp.me/ns#"> <![endif]-->
<!--[if IE 7]><html class="no-js lt-ie9 lt-ie8" lang="en-US" prefix="og: https://ogp.me/ns#"> <![endif]-->
<!--[if IE 8]><html class="no-js lt-ie9" lang="en-US" prefix="og: https://ogp.me/ns#"> <![endif]-->
<!--[if IE 9]><html class="no-js lt-ie10" lang="en-US" prefix="og: https://ogp.me/ns#"> <![endif]-->
<!--[if gt IE 8]><!--><html class="no-js" lang="en-US" prefix="og: https://ogp.me/ns#"> <!--<![endif]--><head><script data-no-optimize="1">var litespeed_docref=sessionStorage.getItem("litespeed_docref");litespeed_docref&&(Object.defineProperty(document,"referrer",{get:function(){return litespeed_docref}}),sessionStorage.removeItem("litespeed_docref"));</script> <meta http-equiv="Content-Type" content="text/html; charset=UTF-8" /><meta name='viewport' content='width=device-width, initial-scale=1, user-scalable=yes' /><link rel="profile" href="https://gmp

Okay so before the next step in the project, Building Language ID, first we need to do one more step i.e HTML Cleaning & Text Extraction. GPT says that this is actually a very tough task and i need to must use the libraries for this task

In [72]:
! pip install beautifulsoup4


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [73]:
! pip install lxml


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [74]:
from bs4 import BeautifulSoup
soup=BeautifulSoup(record["html"],"lxml")

In [75]:
print(soup)

<!DOCTYPE html>
<!--[if lt IE 7]><html class="no-js lt-ie9 lt-ie8 lt-ie7" lang="en-US" prefix="og: https://ogp.me/ns#"> <![endif]--><!--[if IE 7]><html class="no-js lt-ie9 lt-ie8" lang="en-US" prefix="og: https://ogp.me/ns#"> <![endif]--><!--[if IE 8]><html class="no-js lt-ie9" lang="en-US" prefix="og: https://ogp.me/ns#"> <![endif]--><!--[if IE 9]><html class="no-js lt-ie10" lang="en-US" prefix="og: https://ogp.me/ns#"> <![endif]--><!--[if gt IE 8]><!--><html class="no-js" lang="en-US" prefix="og: https://ogp.me/ns#"> <!--<![endif]--><head><script data-no-optimize="1">var litespeed_docref=sessionStorage.getItem("litespeed_docref");litespeed_docref&&(Object.defineProperty(document,"referrer",{get:function(){return litespeed_docref}}),sessionStorage.removeItem("litespeed_docref"));</script> <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/><meta content="width=device-width, initial-scale=1, user-scalable=yes" name="viewport"/><link href="https://gmpg.org/xfn/11" rel="p

In [76]:
for tag in soup(["script","style","noscript","svg","iframe"]):
    tag.decompose()

text=soup.get_text(separator="\n",strip=True)

In [77]:
print(text)

Basketball Shooter - Happy Wheels Game
Happy Wheels
No Result
View All Result
Home
No Result
View All Result
Basketball Shooter
Fullscreen
February 19, 2024
0
4.3/5 - (3 votes)
Table of Contents
Toggle
Game Description
Game Controls
How to Play Basketball Shooter?
Tips and Tricks
Game Developer
Game Platforms
How to Play Unblocked
Conclusion
Game Description
Basketball Shooter
is an addictive online basketball game that challenges players to showcase their accuracy and precision in shooting hoops. Your goal is to make as many successful shots as possible within the given time limit. With its simple yet engaging gameplay, Basketball Shooter is the perfect game for basketball enthusiasts and casual gamers alike.
Game Controls
Mastering the controls in
Basketball Shooter
is key to achieving a high score. The game offers straightforward controls:
Mouse
: Use your mouse to aim and adjust the angle of your shot.
Left Mouse Button
: Click the left mouse button to shoot the basketball.
How to 

Okay wow this is super crazy i've got the clean text, but still it is not that clean for that we will use trafilatura library and we will see how we will obtain the clean text, i am actually very excited in this journey like isn't all of this is crazy like seriously How are these libraries doing the work how did engineers even created these kind of libraries like this is really really crazy i must say

In [78]:
! pip install trafilatura


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Okay there is one more thing that trafilatura accepts the raw html text and the kind of work it does is just crazy i must say

In [79]:
import trafilatura
clean_text=trafilatura.extract(record["html"])
print(clean_text)

Game Description
Basketball Shooter is an addictive online basketball game that challenges players to showcase their accuracy and precision in shooting hoops. Your goal is to make as many successful shots as possible within the given time limit. With its simple yet engaging gameplay, Basketball Shooter is the perfect game for basketball enthusiasts and casual gamers alike.
Game Controls
Mastering the controls in Basketball Shooter is key to achieving a high score. The game offers straightforward controls:
- Mouse: Use your mouse to aim and adjust the angle of your shot.
- Left Mouse Button: Click the left mouse button to shoot the basketball.
How to Play Basketball Shooter?
Playing Basketball Shooter is all about making successful shots and scoring points. Here’s a step-by-step guide on how to play:
- Launch the Game: Start the game by launching it in your web browser.
- Set Your Target: Look at the basketball hoop and decide where you want to make your shot.
- Aim Carefully: Use your 

In [80]:
with open("text_extraction.jsonl","w",encoding="utf-8") as f:
    for record in document_ingestion:
        text=trafilatura.extract(record.pop("html"))
        if text is None:
            continue
        record["text"]=text
        json.dump(record,f,ensure_ascii=False)
        f.write("\n")


Okay so now we are officially done with the Document Ingestion step the next one is Language ID i.e there seems to be a library called as fastText for identifying which language it is so let's just do that

In [81]:
! pip install fastText


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [82]:
import fasttext
model=fasttext.load_model("lid.176.bin")
clean_text=clean_text.replace("\n"," ")
language,confidence=model.predict(clean_text)
print(language)
print(confidence)

('__label__en',)
[0.93511385]


In [83]:
with open("text_extraction.jsonl", "r",encoding="utf-8") as f:
    for line in f:
        new_record=json.loads(line)
        break
        

In [84]:
print(new_record)

{'url': 'http://0530jtss.com/content.asp?id=8198&bg=186', 'timestamp': '2026-06-05T22:41:22Z', 'warc_record_id': '<urn:uuid:bc1d9f66-f584-4630-9648-94691fecba8e>', 'http_status': 200, 'content_type': 'text/html', 'charset': 'gb2312', 'text': '| 通标牌切圆机是专业生产反光标牌圆牌，三角牌的设备，一次成型。是制作交通标牌，小型标牌加工厂 必备的好帮手。\n \u3000\u3000专业生产切圆三角牌弯边一体机 此设备主要用于公路圆标牌、三角警示标牌的制作，一次裁切压边成型，是制作公路反光标牌的专用设备，此设备切圆、压边直径：300-1200mm 三角标牌弯边长度：不限, 可切铝板厚度：1-3mm, 电压 、动力：220V/700W 380V/750W;误 差 率：<1mm;外 形 尺 寸：1200*500*1250mm;重 量：200Kg; 质量保证，欢迎咨询洽谈。 \u3000\u3000河南佳通实业交通设施有限公司专业经营:标牌切圆机,标牌剪板机,液压剪板机、折弯机、覆膜机、剪板覆膜一体机等交通标牌行业专用设备。质量第一,客户至上,诚信 上一篇:交通标牌切圆机\n  下一篇:交通标牌厂专用、交通标志牌切圆设备\n |'}


In [85]:
clear_text=new_record["text"].replace("\n"," ")
language,confidence=model.predict(clear_text)
print(language[0])
print(confidence[0])
new_record["language"]=language[0].replace("__label__"," ").strip()
new_record["language_confidence"]=round(float(confidence[0]),5)
print(new_record)

__label__zh
0.9547341465950012
{'url': 'http://0530jtss.com/content.asp?id=8198&bg=186', 'timestamp': '2026-06-05T22:41:22Z', 'warc_record_id': '<urn:uuid:bc1d9f66-f584-4630-9648-94691fecba8e>', 'http_status': 200, 'content_type': 'text/html', 'charset': 'gb2312', 'text': '| 通标牌切圆机是专业生产反光标牌圆牌，三角牌的设备，一次成型。是制作交通标牌，小型标牌加工厂 必备的好帮手。\n \u3000\u3000专业生产切圆三角牌弯边一体机 此设备主要用于公路圆标牌、三角警示标牌的制作，一次裁切压边成型，是制作公路反光标牌的专用设备，此设备切圆、压边直径：300-1200mm 三角标牌弯边长度：不限, 可切铝板厚度：1-3mm, 电压 、动力：220V/700W 380V/750W;误 差 率：<1mm;外 形 尺 寸：1200*500*1250mm;重 量：200Kg; 质量保证，欢迎咨询洽谈。 \u3000\u3000河南佳通实业交通设施有限公司专业经营:标牌切圆机,标牌剪板机,液压剪板机、折弯机、覆膜机、剪板覆膜一体机等交通标牌行业专用设备。质量第一,客户至上,诚信 上一篇:交通标牌切圆机\n  下一篇:交通标牌厂专用、交通标志牌切圆设备\n |', 'language': 'zh', 'language_confidence': 0.95473}


In [86]:
with open("text_extraction.jsonl","r",encoding="utf-8") as p:
    with open("language_identification.jsonl","w",encoding="utf-8") as f:
        for line in p:
            text_record=json.loads(line)
            text_clean=text_record["text"]
            if text_clean is None:
                continue
            text_clean=" ".join(text_clean.split())
            text_record["text"]=text_clean
            language,confidence=model.predict(text_clean)
            text_record["language"]=language[0].replace("__label__","")
            text_record["language_confidence"]=round(float(confidence[0]),5)
            json.dump(text_record,f,ensure_ascii=False)
            f.write("\n")   

In [87]:
with open("language_identification.jsonl","r",encoding="utf-8") as p:
    with open("english_only.jsonl","w",encoding="utf-8") as f:
        for line in p:
            record=json.loads(line)
            if record["language"]=='en':
                f.write(json.dumps(record,ensure_ascii=False)+"\n")


Okay so i am successfully done with the second step in the project, language ID, where i have identified the language, and the language_confidence using FastText ("lid.176.bin"). Now the next step in the actual project is Deduplication. So GPT says that there are few types of deduplications.
1. Exact Deduplication : Two documents are byte-to-byte identical examples include Hello world and Hello world 
2. Near deduplication : Python is amazing. , Python is amazing!
3. Fuzzy semantic Deduplication : Cats are mammals, Domestic cats belong to the mammal family.

Okay for this particular pipeline, GPT says that it's better to do Exact Deduplication first and then next Near deduplication, Let's get into everything in detail like let's understand what's happening exactly what we will do in exact deduplication and what we will do in near duplication so let's understand these in detail and then we will get into the next step, Quality filtering.

Now let's understand the exact deduplication first and how we do it. There are few algorithms for it but let's use SHA-256 algorithm.

In [91]:
import hashlib
text="A cat sat on a mat"
hashlib.sha256(text.encode("utf-8")).hexdigest()

'a0f9826e2867884bbaf520f5f935bfa41e849cc8420042c354788dac66d123f2'

In [92]:
with open("language_identification.jsonl","r",encoding="utf-8") as f:
    for line in f:
        text_record=json.loads(line)
        actual_text=text_record["text"]
        break

hashlib.sha256(actual_text.encode("utf-8")).hexdigest()

'869d634da8e4bb602549c28321db369d847cf113c78ecd3f755ad56ab15b8ae0'

In [93]:
with open("exact_deduplication.jsonl","w",encoding="utf-8") as p:
    seen=set()
    i=0
    j=0
    with open("english_only.jsonl","r",encoding="utf-8") as f:
        for line in f:
            i+=1
            text_record=json.loads(line)
            actual_text=text_record.get("text")
            if not actual_text :
                continue
            text_hash=hashlib.sha256(actual_text.encode("utf-8")).hexdigest()
            if text_hash not in seen:
                seen.add(text_hash)
                json.dump(text_record,p,ensure_ascii=False)
                p.write("\n")
                j+=1

print(f"english_only.jsonl contains {i} records")
print(f"exact_deduplication.jsonl contains {j} records")
print(f" Number of deduplicate records removed : {i-j}")



english_only.jsonl contains 8367 records
exact_deduplication.jsonl contains 8140 records
 Number of deduplicate records removed : 227


Like seeing these numbers is just crazy there were 227 deduplicates which we removed so deduplication rate is 227/8367 that's approximately 2.71% But these 2.71% documents are exactly similar to other they are same byte-to-byte. Next we will perform the near-deduplication to remove more similar documents let's see how to do the near deduplication next so let's go.

Okay GPT says there are a couple of methods for near deduplication

1. Fuzzy string method ( This is not for LLM Scale like they basically compare two documents based on some similarity but this is not generally used at the scale of modern LLMs.)
2. Shingling (Instead of treating a document as one long string, break into overlapping chunks) for example The cat sat on the mat 3-word shingles become {The cat sat}, {cat sat on} etc something like this.Now each document is represented as a shingle and two documents are similar if they share many shingles.
3. Min Hash This is the important algorithm i should implement says GPT. Instead of storing million shingles for every document, MiniHash creates a small signature (often 128 or 256 integers). So Document -> Shingles -> Hash every Shingle -> Keep only minimun Hash values -> MinHash signature
Similar documents have similar MinHash signatures.
4. Locality Sensitive Hashing (LSH): MinHash tells us the similarity, LSH tells us which documents are worth comparing. Without LSH we need to compare every two documents. Instead Locality Sensitive Hashing does something like this: MinHash -> Split signature into bands -> Only documents in the same bucket -> Compare those.
GPT says that almost every large-scale pipeline combines shingling -> MinHash -> LSH.

Actually i may have to implement the Shingling, MinHash, LSH by myself but for now i want to use the libraries only.

In [94]:
! pip install datasketch


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
def shingles(text,k=5):
    words=text.split()
    return {" ".join(words[i:i+k]) for i in range(len(words)-k+1)}


In [6]:
print(shingles("The cat sat on a mat"))

{'cat sat on a mat', 'The cat sat on a'}


In [7]:
with open("exact_deduplication.jsonl","r",encoding="utf-8") as f:
    for line in f:
        record=json.loads(line)

In [8]:
record_shingles=shingles(record["text"])
print(record_shingles)

{'them to the main Demozoo', 'production, brought to you by', 'sceners » Untitled Release date:', 'Cyberpunks Unity and Inward Download', 'Download (SCL) Download A Gasman', '1st at DiHalt 2006 ZX', 'you by the Demozoo network.', '» Untitled 2006 Cyberpunks Unity', 'Download Download (SCL) Download A', 'the Demozoo network. Corrections, additions,', 'Demo competition Untitled on Demozoo', 'brought to you by the', 'Demozoo network. Corrections, additions, new', 'Untitled on Demozoo » Untitled', 'articles » releases » parties', '29 April 2006 1st at', 'competition Untitled on Demozoo »', 'main Demozoo site ! Grab', '» sceners » Untitled Release', 'ZX Spectrum Demo competition Untitled', 'by the Demozoo network. Corrections,', 'Untitled 2006 Cyberpunks Unity and', '» releases » parties »', 'parties » sceners » Untitled', 'A Gasman production, brought to', 'Unity and Inward Download Download', 'releases? Submit them to the', '! Grab the Demotopia newsfeed', 'Grab the Demotopia newsfeed at:

In [9]:
from datasketch import MinHash
mh=MinHash(num_perm=128)
for shingle in record_shingles:
    mh.update(shingle.encode('utf-8'))

In [10]:
print(mh)

In [11]:
print(mh.hashvalues)

[  4605717  29275991  14144481  24303592  25341538 248722312  38947453
 127484403   2698476  47250973  74919660 117903973  37225934  25464332
  49643803  32495481 288936676 108630764  20657622  95096459  24405241
   8220507 167238532 136996681 199274861  17347814   3489237  18436953
  63257922  43047482  15310555 339986889  14014737  99266490  73791224
  74875119  31664800  59125628  58649020  18964536   6399697   1216951
  99217257  23841799  46241774  11964271    419601  14778302  91710573
  72211661  53031162  11169543   2397770  39326614  52270927    788555
  51372099 344304446 197290209 106271339 292400818  45199797 208464140
 210398741 115771221  69917193   8306607 117804507  73487337  27919320
  67034164  30277014  16705366  53844679  49850399  14624395  34303839
   5789500 150861940 148150148  54775131  65756241  17075755   9948581
  16335582   5509341   7990631   1480371  50420057  41719952  96662837
  43412933 262840580  73842132   5480054   4201774   9156052  19749942
 16325

In [12]:

minihashes={}
with open("exact_deduplication.jsonl","r",encoding="utf-8") as f:
    for i,line in enumerate(f):
        record=json.loads(line)
        mh=MinHash(num_perm=128)
        record_shingles=shingles(record["text"])
        for shingle in record_shingles:
            mh.update(shingle.encode('utf-8'))
        minihashes[f"record_{i}"]=mh
        lsh.insert(f"record_{i}",mh)

    

In [104]:
a=[1]
b=[]
b+=a[1:]
print(b)

[]


In [13]:

with open("near_deduplication.jsonl","w",encoding="utf-8") as f:
    with open("exact_deduplication.jsonl","r",encoding="utf-8") as p:
        near_duplicates=set()
        num_elements_initial=0
        num_elements_final=0
        for i,line in enumerate(p):
            num_elements_initial+=1
            index=f"record_{i}"
            if index in near_duplicates:
                continue
            f.write(line)
            num_elements_final+=1
            candidates=lsh.query(minihashes[index])
            for candidate in candidates:
                if candidate ==index:
                    continue
                similarity=minihashes[index].jaccard(minihashes[candidate])
                if similarity>=0.8:
                    near_duplicates.add(candidate)


print(f"No of records in exact_deduplication.jsonl is {num_elements_initial}")
print(f"No of records in near_deduplication.jsonl is {num_elements_final}")
print(f"No of near deduplicates records found is {num_elements_initial-num_elements_final}")

            
            


No of records in exact_deduplication.jsonl is 8140
No of records in near_deduplication.jsonl is 8063
No of near deduplicates records found is 77


Okay so i am also done with the near deduplication and the result is we left with 8045 records finally. Next step in the process is Quality filtering. What do i need to exactly do in the quality filtering step, i am done with document ingestion, Trafilaturation, language identification, exact dedup, near dedup. I understand that i am gonna filter quality text but how the hell am i gonna do it exactly? There are kind of three stages, the first one is Quality filtering, second is Toxicity or policy filteration and the next is PII detection. For Quality filtering, we will follow the RefinedWeb heuristics. So let's visit RefinedWeb and follow the rules from there

First i have actually started with 20388 documents in total and after only extracting the english documents i am left with 8367 documents, after exact dedup i am actually left with 8140 documents and after Near dedup i am finally now left with 8045 documents, now i am doing the Quality filtering stage and i am gonna implement the following quality filters:
1.Minimum Document length
2.Maximum Document length
3.Average word length
4.Stopword ratio
5.Symbol ratio
6.Digit ratio
7.Vocabulary diversity
8.Charecter repetition
9.Line repetition
10.Duplicate sentence ratio
11.Duplicate paragraph ration
12.Maximum line length
13.Uppercase ratio
14.Punctuation ratio

Actually GPT asked me to implement these

In [14]:
with open("near_deduplication.jsonl","r",encoding='utf-8') as f:
    for i,line in enumerate(f):
        if i==6000:
            record=json.loads(line)
            print(record["text"])

| Source Id | NCBI (Entrez) Gene Id | Gene Symbol | Gene Description | | ENSG00000075886 | 113457 | TUBA3D | tubulin alpha 3d [Source:HGNC Symbo... | | ENSG00000101162 | 81027 | TUBB1 | tubulin beta 1 class VI [Source:HGN... | | ENSG00000104833 | 10382 | TUBB4A | tubulin beta 4A class IVa [Source:H... | | ENSG00000115484 | 10575 | CCT4 | chaperonin containing TCP1 subunit ... | | ENSG00000120438 | 6950 | TCP1 | t-complex 1 [Source:HGNC Symbol;Acc... | | ENSG00000123416 | 10376 | TUBA1B | tubulin alpha 1b [Source:HGNC Symbo... | | ENSG00000127824 | 7277 | TUBA4A | tubulin alpha 4a [Source:HGNC Symbo... | | ENSG00000132141 | 10693 | CCT6B | chaperonin containing TCP1 subunit ... | | ENSG00000135624 | 10574 | CCT7 | chaperonin containing TCP1 subunit ... | | ENSG00000137267 | 7280 | TUBB2A | tubulin beta 2A class IIa [Source:H... | | ENSG00000137285 | 347733 | TUBB2B | tubulin beta 2B class IIb [Source:H... | | ENSG00000146731 | 908 | CCT6A | chaperonin containing TCP1 subunit ... | | ENS

In [15]:
print(record["text"].split())

['|', 'Source', 'Id', '|', 'NCBI', '(Entrez)', 'Gene', 'Id', '|', 'Gene', 'Symbol', '|', 'Gene', 'Description', '|', '|', 'ENSG00000075886', '|', '113457', '|', 'TUBA3D', '|', 'tubulin', 'alpha', '3d', '[Source:HGNC', 'Symbo...', '|', '|', 'ENSG00000101162', '|', '81027', '|', 'TUBB1', '|', 'tubulin', 'beta', '1', 'class', 'VI', '[Source:HGN...', '|', '|', 'ENSG00000104833', '|', '10382', '|', 'TUBB4A', '|', 'tubulin', 'beta', '4A', 'class', 'IVa', '[Source:H...', '|', '|', 'ENSG00000115484', '|', '10575', '|', 'CCT4', '|', 'chaperonin', 'containing', 'TCP1', 'subunit', '...', '|', '|', 'ENSG00000120438', '|', '6950', '|', 'TCP1', '|', 't-complex', '1', '[Source:HGNC', 'Symbol;Acc...', '|', '|', 'ENSG00000123416', '|', '10376', '|', 'TUBA1B', '|', 'tubulin', 'alpha', '1b', '[Source:HGNC', 'Symbo...', '|', '|', 'ENSG00000127824', '|', '7277', '|', 'TUBA4A', '|', 'tubulin', 'alpha', '4a', '[Source:HGNC', 'Symbo...', '|', '|', 'ENSG00000132141', '|', '10693', '|', 'CCT6B', '|', 'chaperoni

In [16]:
! pip install --upgrade pip


In [17]:
! pip install nltk

In [ ]:
#01 For minimum Document length i am using word count as 50 words.(Dolma)
def min_document_length(text):
    words=text.split()
    if len(words)<50:
        return False
    else:
        return True
    
#02 For maximum Document length i am using word as 1,00,000 words. (Dolma)
def max_document_length(text):
    words=text.split()
    if len(words)<100000:
        return True
    else:
        return False
    
#03 For Average word length i am using minimum average word length must be greater than or equal to 3 and maximum average word length must be less than or equal to 10. (dolma)
def average_word_length(text):
    words=text.split()
    store=[]
    for word in words:
        store.append(len(word))
    avg_len=sum(store)/len(store)
    if avg_len>=3 and avg_len<=10:
        return True
    else:
        return False
    
#04 For stopword ratio i am using two rules, the first one is that if the document has atleast two distinct words from given list and the seond rule is that if stopword ratio is >0.20
import nltk
nltk.download('stopwords')
nltk.download("punkt_tab")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
english_stopwords=set(stopwords.words('english'))
def stopword_ratio(text):
    words=word_tokenize(text.lower())
    my_set={'the','be','to','of','and','that','have','with'}
    doc_set=set()
    for word in words:
        if word in my_set:
            doc_set.add(word)
    if len(doc_set)<2:
        return False
    if len(doc_set)>=2:
        total_word_count=len(words)
        stopwords_found=[w for w in words if w in english_stopwords]
        stopwords_count=len(stopwords_found)
        stopword_ratio=stopwords_count/total_word_count
        if stopword_ratio<=0.20:
            return False
        else:
            return True
        
#05 For symbol ratio, if >=0.10 then discard or else keep it
import re
def symbol_ratio(text):
    pattern=r'\.\.\.|…|\w+|[^\w\s]'
    splitted_text=re.findall(pattern,text)
    symbol_ratio=(splitted_text.count('#')+splitted_text.count('...')+splitted_text.count('…'))/len(splitted_text)
    if symbol_ratio>=0.1:
        return False
    else:
        return True
    
#06 For digit ratio if >=0.20 then discard or else keep it
def digit_ratio(text):
    digit_count=sum(char.isdigit() for char in text)
    digit_ratio=digit_count/len(text)
    if digit_ratio<0.20:
        return True
    else:
        return False
    

    




            

    

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/vedasamhith/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/vedasamhith/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [19]:
import json
with open ('min_document_length.jsonl','w',encoding='utf-8') as p:
    with open('near_deduplication.jsonl','r') as f:
        i=0
        j=0
        for line in f:
            j+=1
            record=json.loads(line)
            if min_document_length(record["text"]) is True:
                i+=1
                p.write(line)
                

print(f"total number of records before min_document_length is {j} ")
print(f"total number of records after min_document_length is {i}")

    


total number of records before min_document_length is 8063 
total number of records after min_document_length is 6743


In [20]:
with open('min_document_length.jsonl','r') as f:
    i=0
    j=0
    for line in f:
        j+=1
        record=json.loads(line)
        if max_document_length(record["text"]) is True:
            i+=1

print(f"total number of records before max_document_length is {j}")
print(f"total number of records after max_document_length is {i}")

total number of records before max_document_length is 6743
total number of records after max_document_length is 6743


In [21]:
with open("average_word_length.jsonl","w",encoding='utf-8') as p:
    with open('min_document_length.jsonl','r') as f:
        i=0
        j=0
        for line in f:
            j+=1
            record=json.loads(line)
            if average_word_length(record["text"]) is True:
                i+=1
                p.write(line)

print(f"total number of records before average_word_length is {j}")
print(f"total number of records after average_word_length is {i}")

total number of records before average_word_length is 6743
total number of records after average_word_length is 6647


In [22]:
with open("stop_word_ratio.jsonl","w",encoding='utf-8') as p:
    with open("average_word_length.jsonl","r") as f:
         i=0
         j=0
         for line in f:
            j+=1
            record=json.loads(line)
            if stopword_ratio(record["text"]) is True:
                i+=1
                p.write(line)

print(f"total number of records before stop_word_ratio is {j}")
print(f"total number of records after stop_word_ratio is {i}")


total number of records before stop_word_ratio is 6647
total number of records after stop_word_ratio is 4763


In [23]:
with open("symbol_ratio.jsonl","w",encoding='utf-8') as p:
    with open("stop_word_ratio.jsonl","r") as f:
         i=0
         j=0
         for line in f:
            j+=1
            record=json.loads(line)
            if symbol_ratio(record["text"]) is True:
                i+=1
                p.write(line)

print(f"total number of records before symbol_ratio is {j}")
print(f"total number of records after symbol_ratio is {i}")

total number of records before symbol_ratio is 4763
total number of records after symbol_ratio is 4761


In [24]:
with open("digit_ratio.jsonl","w",encoding='utf-8') as p:
    with open("symbol_ratio.jsonl","r") as f:
         i=0
         j=0
         for line in f:
            j+=1
            record=json.loads(line)
            if digit_ratio(record["text"]) is True:
                i+=1
                p.write(line)

print(f"total number of records before digit_ratio is {j}")
print(f"total number of records after digit_ratio is {i}")

total number of records before digit_ratio is 4761
total number of records after digit_ratio is 4758


In [25]:
! pip install detoxify

In [26]:
from detoxify import Detoxify
model=Detoxify('original')
with open("digit_ratio.jsonl","r",encoding='utf-8') as f:
    for i,line in enumerate(f):
        if i==3000:
            record=json.loads(line)
            print(model.predict(record["text"]))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'toxicity': 0.0012960162, 'severe_toxicity': 0.00010673835, 'obscene': 0.00020208377, 'threat': 0.0001264014, 'insult': 0.00020383568, 'identity_attack': 0.0001596619}


In [27]:
with open("block.6edde7d0cf.jpgq6j.txt") as f:
    adult_domains={line.strip().lower() for line in f}


In [28]:
import json
from urllib.parse import urlparse
i=0
j=0
with open("url_filtering_jsonl","w",encoding='utf-8') as p:
    with open("digit_ratio.jsonl","r",encoding='utf-8') as f:
        for line in f:
            i+=1
            record=json.loads(line)
            domain=urlparse(record["url"]).hostname.lower()
            if domain.startswith("www."):
                domain=domain[4:]
            if domain not in adult_domains:
                j+=1
                p.write(line)

print(f"Number of records before url_filtering {i}")
print(f"Number of records after url filtering {j}")


Number of records before url_filtering 4758
Number of records after url filtering 4718


In [29]:
i=0
j=0
with open("toxicity_filtering_jsonl","w",encoding='utf-8') as p:
    with open("url_filtering_jsonl","r",encoding='utf-8') as f:
        for line in f:
            i+=1
            record=json.loads(line)
            scores=model.predict(record["text"])
            if scores["toxicity"]>=0.85 or scores["severe_toxicity"]>=0.50 or scores["threat"]>=0.50 or scores["identity_attack"]>=0.60:
                continue
            else:
                j+=1
                p.write(line)
            
print(f"Number of records before toxicity filtering {i}")
print(f"Number of records after toxicity filtering {j}")


Number of records before toxicity filtering 4718
Number of records after toxicity filtering 4712


In [30]:
i=0
j=0
with open("toxicity_double_filtering.jsonl","w",encoding='utf-8') as p:
    with open("toxicity_filtering.jsonl","r",encoding='utf-8') as f:
         for line in f:
            i+=1
            record=json.loads(line)
            scores=model.predict(record["text"])
            if scores["toxicity"]>=0.50 or scores["severe_toxicity"]>=0.50 or scores["threat"]>=0.50 or scores["identity_attack"]>=0.60:
                continue
            else:
                j+=1
                p.write(line)


print(f"Number of records before double toxicity filtering 4704")
print(f"Number of records after double toxicity filtering 4700")

Number of records before double toxicity filtering 4704
Number of records after double toxicity filtering 4700


In [8]:
! pip install presidio-analyzer presidio-anonymizer

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 1.8 MB/s  0:00:01 eta 0:00:01
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 2.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 4.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 658.5/658.5 kB 755.4 kB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.9/791.9 kB 1.4 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 3.6 MB/s  0:00:02m0:00:0100:01
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.1
    Uninstalling typing-inspection-0.4.1:

In [9]:
! python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.7 MB/s  0:00:07m0:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [31]:
from presidio_analyzer import AnalyzerEngine
analyzer=AnalyzerEngine()
text="Contact me at john@gmail.com"
results=analyzer.analyze(text=text,language="en")
print(results)

[type: EMAIL_ADDRESS, start: 14, end: 28, score: 1.0, type: URL, start: 19, end: 28, score: 0.5]


Keep these :
{"EMAIL_ADDRESS","PHONE_NUMBER","CREDIT_CARD","IP_ADDRESS","US_SSN","IBAN_CODE","US_PASSPORT","US_DRIVER_LICENSE"}

In [32]:
i=0
j=0
with open("PII.jsonl","w",encoding='utf-8') as p:
    with open("toxicity_double_filtering.jsonl","r",encoding='utf-8') as f:
        for line in f:
            i+=1
            record=json.loads(line)
            results=analyzer.analyze(text=record["text"],language="en")
            ALLOWED={"EMAIL_ADDRESS","PHONE_NUMBER","CREDIT_CARD","IP_ADRESS","US_SSN","IBAN_CODE","US_PASSPORT","US_DRIVER_LICENSE"}
            has_pii=any(r.entity_type in ALLOWED and r.score>=0.8 for r in results)
            if not has_pii:
                j+=1
                p.write(line)
            

print(f"Before PII detection {i}")
print(f"After PII detection {j}")

Before PII detection 4700
After PII detection 4444


In [34]:
i=0
j=0
with open("final_english_filtering.jsonl","w",encoding='utf-8') as p:
    with open("PII.jsonl","r",encoding='utf-8') as f:
        for line in f:
            i+=1
            record=json.loads(line)
            if record["language_confidence"]>=0.80:
                j+=1
                p.write(line)

print(f"before {i}")
print(f"after {j}")


before 4444
after 4203
